<a href="https://colab.research.google.com/github/sevenjunebaby/ML/blob/main/Homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
# Load data
from google.colab import files
uploaded = files.upload()
import io
df = pd.read_csv('/content/Housing.csv')

Saving Housing.csv to Housing.csv


In [3]:
print("\n" + "=" * 80)
print("2. ANALYSE EXPLORATOIRE DES DONNÉES (EDA)")
print("=" * 80)
print(f"\n📈 Statistiques descriptives:")
print(df.describe())
print(f"\n🔍 Types de données:")
print(df.dtypes)
print(f"\n❌ Valeurs manquantes:")
print(df.isnull().sum())
print(f"\n📊 Informations sur les variables catégoriques:")
for col in df.select_dtypes(include='object').columns:
   print(f" {col}: {df[col].unique()}")


2. ANALYSE EXPLORATOIRE DES DONNÉES (EDA)

📈 Statistiques descriptives:
              price          area    bedrooms   bathrooms     stories  \
count  5.450000e+02    545.000000  545.000000  545.000000  545.000000   
mean   4.766729e+06   5150.541284    2.965138    1.286239    1.805505   
std    1.870440e+06   2170.141023    0.738064    0.502470    0.867492   
min    1.750000e+06   1650.000000    1.000000    1.000000    1.000000   
25%    3.430000e+06   3600.000000    2.000000    1.000000    1.000000   
50%    4.340000e+06   4600.000000    3.000000    1.000000    2.000000   
75%    5.740000e+06   6360.000000    3.000000    2.000000    2.000000   
max    1.330000e+07  16200.000000    6.000000    4.000000    4.000000   

          parking  
count  545.000000  
mean     0.693578  
std      0.861586  
min      0.000000  
25%      0.000000  
50%      0.000000  
75%      1.000000  
max      3.000000  

🔍 Types de données:
price                int64
area                 int64
bedrooms      

In [4]:
data = df[['area', 'price']]
print(data.shape)
X = data[['area']].values
y = data['price'].values
print(X.shape, y.shape)

(545, 2)
(545, 1) (545,)


In [5]:
from sklearn.preprocessing import StandardScaler
scaler_X = StandardScaler()
scaler_y = StandardScaler()

In [6]:
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()
print(X_scaled.shape, y_scaled.shape)

(545, 1) (545,)


In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X_scaled, y_scaled, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(436, 1) (109, 1)


In [11]:
from sklearn.tree import DecisionTreeRegressor

t = DecisionTreeRegressor().fit(X_train, y_train)


In [12]:
y_pred_scaled = t.predict(X_test)

In [13]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred_scaled)
print("Score R2 :", r2)

Score R2 : 0.27479960296819683


In [14]:
numeric_features = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
binary_features = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
categorical_features = ['furnishingstatus']
target = 'price'

In [17]:
for col in binary_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
# pour encoder les variables qui ont des valeurs de yes/no

In [18]:
# Encodage de furnishingstatus avec one-hot encoding
df = pd.get_dummies(df, columns=categorical_features, drop_first=True)

In [19]:
# regroupement de tout les caracteristiques encodees/standarisees
encoded_features = df.columns.difference([target]).tolist()
X = df[encoded_features].values
y = df[target].values

In [20]:
# Standardisation des features numériques uniquement
scaler_X = StandardScaler()
# Créer une copie de X pour préserver les colonnes non numériques
X_scaled = X.copy()
# Standardiser uniquement les colonnes numériques (indices correspondants)
df[numeric_features] = scaler_X.fit_transform(df[numeric_features])
# Standardisation de y (facultatif, inclus ici)
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
X_scaled, y_scaled, test_size=0.2, random_state=42)
print("Forme de X_train :", X_train.shape)
print("Forme de X_test :", X_test.shape)

Forme de X_train : (436, 13)
Forme de X_test : (109, 13)


In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
param_grid = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': [None, 'sqrt', 'log2']
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid.fit(X_train, y_train)
print("Best R2 (cross-validation):", grid.best_score_)


Best R2 (cross-validation): 0.5388360837048789


In [34]:
y_pred_scaled = grid.predict(X_test)

In [35]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred_scaled)
print("Score R2 :", r2)

Score R2 : 0.4870843656299553
